# 🚢 Titanic Classification — قدم‌به‌قدم برای حمید

حمید، در این Notebook یاد می‌گیری با اطلاعات یک مسافر، زنده‌ماندن او را پیش‌بینی کنی.

**روش اجرا:** سلول‌های Code را از بالا به پایین با **Shift + Enter** اجرا کن. سلول‌های Markdown توضیح هستند. اول تمرین را امتحان کن، بعد راه‌حل سلول بعدی را بخوان. برای شروع دوباره، محیط اجرا را Restart کن و همهٔ سلول‌ها را به‌ترتیب اجرا کن.

مسیر یادگیری: **data → X/y → train/test split → create model → fit → predict → evaluate → new passenger**

فایل محلی لازم نیست؛ اگر `titanic.csv` موجود نباشد، داده از اینترنت خوانده می‌شود. در Kaggle برای این حالت Internet را روشن کن. اگر اینترنت نداری، CSV را اضافه کن و مسیر آن را در سلول بارگذاری تغییر بده.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hghaderi-54/IntroAI/blob/main/Titanic/beginner/Titanic_Classification_Beginner_FA.ipynb)

**بازکردن و اجرا:** این دکمه دفترچه را از GitHub در Colab باز می‌کند. همچنین می‌توانی فایل را مستقیم در Colab باز کنی یا در Kaggle از گزینهٔ Import/Upload Notebook استفاده کنی. GitHub فایل را نمایش می‌دهد؛ اجرای کد در محیط Notebook انجام می‌شود.

## ۱. Classification یعنی چه؟

در Regression یک عدد مثل قیمت خانه پیش‌بینی می‌کردیم. در **Classification (طبقه‌بندی)** یک دسته را پیش‌بینی می‌کنیم.

ستون `Survived` جواب واقعی هر مسافر است:
- `0`: زنده نمانده است.
- `1`: زنده مانده است.

این عددها برچسب دسته‌اند. مدل قرار نیست نمره یا قیمت پیش‌بینی کند. این تمرین آموزشی است؛ نتیجهٔ مدل دربارهٔ یک فرد، قطعی نیست.

✏️ **تمرین:** پیش‌بینی «نمرهٔ امتحان» و «قبول/رد» کدام‌یک Classification است؟

<details><summary>راه‌حل</summary>قبول/رد Classification است؛ پیش‌بینی نمره Regression است.</details>

## ۲. ابزارها و data

`pandas` داده را به شکل جدول نگه می‌دارد؛ `DataFrame` نام این جدول است. `scikit-learn` ابزار ساخت و ارزیابی مدل را دارد. این کتابخانه‌ها معمولاً در Colab و Kaggle آماده‌اند.

اگر `ModuleNotFoundError` دیدی، یک سلول اضافه کن و `%pip install pandas scikit-learn matplotlib` را اجرا کن، سپس از بالا شروع کن.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/SharifiZarchi/IntroAI/main/Session_05/Titanic/titanic.csv"
LOCAL_CSV = Path("titanic.csv")  # In Kaggle, you can use /kaggle/input/.../titanic.csv
source = LOCAL_CSV if LOCAL_CSV.exists() else DATA_URL
df = pd.read_csv(source)
print("Data source:", source)
display(df.head())

## ۳. یک نگاه ساده به داده

هر **row (ردیف)** یک مسافر است؛ هر **column (ستون)** یک ویژگی یا جواب را نگه می‌دارد. فعلاً فقط سه **feature (ویژگی ورودی)** داریم:
- `Pclass`: کلاس بلیت، با مقدارهای ۱، ۲ یا ۳؛ عدد کوچک‌تر کلاس بالاتر است.
- `Sex`: جنسیت ثبت‌شده در داده.
- `Age`: سن مسافر.

`Survived` همان **target (جوابی که می‌خواهیم پیش‌بینی کنیم)** است. ستون‌هایی مثل نام و شمارهٔ بلیت را برای ساده‌ماندن تمرین استفاده نمی‌کنیم.

`shape` تعداد ردیف و ستون را نشان می‌دهد. `isna().sum()` تعداد مقدارهای خالی هر ستون را می‌شمارد.

In [ ]:
print("Rows, columns:", df.shape)
display(df[["Pclass", "Sex", "Age", "Survived"]].head())
print("Missing values:")
print(df[["Pclass", "Sex", "Age", "Survived"]].isna().sum())
print("Passenger counts by label:")
print(df["Survived"].value_counts().sort_index())

In [ ]:
df["Survived"].value_counts().sort_index().plot.bar(color=["gray", "teal"], rot=0)
plt.xticks([0, 1], ["0: Did not survive", "1: Survived"])
plt.ylabel("Passenger count")
plt.title("Titanic survival labels")
plt.show()

✏️ **تمرین:** با `head(3)` فقط سه مسافر اول را نشان بده. کدام ستون انتخاب‌شده مقدار خالی دارد؟ قبل از رفتن به سلول بعدی تلاش کن.

In [ ]:
display(df.head(3))
print(df[["Pclass", "Sex", "Age", "Survived"]].isna().sum())

## ۴. آماده‌سازی ساده و ساخت X/y

مدل ورودی عددی می‌خواهد. **Encoding (کدگذاری)** یعنی تبدیل دسته به عدد. اینجا `female` را ۰ و `male` را ۱ می‌کنیم. این فقط یک قرارداد است و نشانهٔ برتری نیست. آن را در ستون جدید `Sex_encoded` ذخیره می‌کنیم.

برای سادگی، ردیفی که جواب یا کلاس بلیت یا جنسیت ندارد کنار می‌گذاریم. سن‌های خالی را بعد از train/test split پر می‌کنیم.

In [ ]:
data = df[["Pclass", "Sex", "Age", "Survived"]].copy()
data["Sex_encoded"] = data["Sex"].map({"female": 0, "male": 1})
data = data.dropna(subset=["Pclass", "Sex_encoded", "Survived"])
display(data.head())

**چه اطلاعاتی داریم؟ → X**: کلاس بلیت، جنسیت کدگذاری‌شده و سن.

**چه چیزی می‌خواهیم بدانیم؟ → y**: زنده‌ماندن.

`X` یک جدول **2-D (دوبعدی)** است: محور اول مسافرها، محور دوم ویژگی‌ها. حتی اگر فقط سن را انتخاب کنیم، مدل جدول «تعداد مسافر × یک ویژگی» می‌خواهد.

در `df[["Age"]]`، براکت داخلی یک **list از نام ستون‌ها** می‌سازد و براکت بیرونی آن ستون‌ها را از جدول انتخاب می‌کند؛ نتیجه DataFrame است. ولی `df["Age"]` یک **Series (ستون یک‌بعدی)** می‌دهد.

`y` برای این مسئله فقط یک جواب برای هر مسافر دارد، پس **1-D (یک‌بعدی)** است. `Survived` را داخل X نمی‌گذاریم؛ آن جواب سؤال است!

✏️ **تمرین:** X را از `Pclass`، `Sex_encoded` و `Age` بساز و y را از `Survived`. راه‌حل در سلول بعدی است.

In [ ]:
feature_columns = ["Pclass", "Sex_encoded", "Age"]
X = data[feature_columns].copy()
y = data["Survived"].astype(int)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("One feature as a table:", data[["Age"]].shape)
print("One column as a Series:", data["Age"].shape)

اگر n مسافر داشته باشیم، شکل X برابر `(n, 3)` و شکل y برابر `(n,)` است. کامای داخل `(n,)` یعنی فقط یک اندازه داریم؛ y ستون دوم مخفی ندارد.

## ۵. Train/Test Split — آموزش و امتحان

۸۰٪ داده برای **train (آموزش)** و ۲۰٪ برای **test (امتحان)** کنار می‌گذاریم. مدل نباید هنگام آموزش جواب‌های test را ببیند.

`random_state=42` تقسیم تصادفی را در اجرای دوباره ثابت نگه می‌دارد؛ ۴۲ عدد جادویی نیست. `stratify=y` نسبت دسته‌های ۰ و ۱ را در دو بخش تقریباً حفظ می‌کند.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test = X_test.copy()
print("Training:", X_train.shape, y_train.shape)
print("Testing:", X_test.shape, y_test.shape)

### سن‌های خالی را پر کنیم

**Median (میانه)** مقدار وسط سن‌ها پس از مرتب‌کردن است. آن را فقط از train حساب می‌کنیم و همان عدد را برای train، test و مسافر جدید نگه می‌داریم. این کار مانع استفاده از اطلاعات test در آماده‌سازی مدل می‌شود؛ به این اشتباه **data leakage (نشت اطلاعات)** می‌گویند.

In [ ]:
age_median = X_train["Age"].median()
X_train["Age"] = X_train["Age"].fillna(age_median)
X_test["Age"] = X_test["Age"].fillna(age_median)
print("Training age median:", age_median)
print("Missing training values:", X_train.isna().sum().sum())
print("Missing test values:", X_test.isna().sum().sum())

## ۶. Create Model — ساخت مدل

از **LogisticRegression** استفاده می‌کنیم. با وجود کلمهٔ Regression در نامش، این مدل برای Classification مناسب است. مدل رابطهٔ ورودی‌ها با دستهٔ ۰/۱ را یاد می‌گیرد.

این خط فقط مدل را می‌سازد؛ هنوز چیزی یاد نگرفته است. `max_iter=1000` به فرایند یادگیری فرصت کافی می‌دهد.

In [ ]:
model = LogisticRegression(max_iter=1000)

## ۷. Fit — یادگیری

`fit(X_train, y_train)` یعنی «این ویژگی‌ها و جواب‌های درستشان را ببین و یاد بگیر». برای آموزش فقط از train استفاده می‌کنیم.

✏️ **تمرین:** در `model.fit(____, ____)` چه چیزهایی قرار می‌گیرند؟

In [ ]:
model.fit(X_train, y_train)
print("Model trained!")

## ۸. Predict — پیش‌بینی

حالا `predict(X_test)` جواب مسافرهایی را حدس می‌زند که برای آموزش استفاده نشده‌اند. اینجا y_test را به مدل نمی‌دهیم؛ آن را برای بررسی جواب‌ها نگه می‌داریم.

In [ ]:
y_pred = model.predict(X_test)
comparison = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
display(comparison.head(10))

## ۹. Evaluate — بررسی نتیجه

**Accuracy (دقت کلی)** یعنی تعداد پیش‌بینی‌های درست تقسیم بر تعداد همهٔ پیش‌بینی‌ها. مثلاً ۸ درست از ۱۰ یعنی ۸۰٪.

برای مقایسه، یک حدس خیلی ساده هم می‌سازیم: همیشه پرتکرارترین جواب **train** را بگو. مدل مفید باید نسبت به این معیار ساده بررسی شود. Accuracy به‌تنهایی همهٔ خطاها را توضیح نمی‌دهد.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
majority_label = int(y_train.mode().iloc[0])
baseline_predictions = [majority_label] * len(y_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
print(f"Model accuracy: {accuracy:.1%}")
print(f"Always predict {majority_label}: {baseline_accuracy:.1%}")

**Confusion Matrix (ماتریس خطا)** نشان می‌دهد کدام جواب‌ها درست و کدام اشتباه بوده‌اند.

در نمایش زیر، **ردیف‌ها جواب واقعی** و **ستون‌ها پیش‌بینی** هستند؛ ترتیب هر دو ۰ سپس ۱ است:
- بالا چپ: واقعاً ۰، پیش‌بینی ۰؛ درست.
- بالا راست: واقعاً ۰، پیش‌بینی ۱؛ اشتباه.
- پایین چپ: واقعاً ۱، پیش‌بینی ۰؛ اشتباه.
- پایین راست: واقعاً ۱، پیش‌بینی ۱؛ درست.

عددهای قطر اصلی درست‌اند. عددهای خارج قطر خطا هستند.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=[0, 1]).plot(cmap="Blues", colorbar=False)
plt.title("Rows: actual | Columns: predicted")
plt.show()
print("Correct predictions:", cm[0, 0] + cm[1, 1])
print("Wrong predictions:", cm[0, 1] + cm[1, 0])

✏️ **تمرین:** با جمع عددهای روی قطر و تقسیم بر جمع کل ماتریس، Accuracy را دوباره حساب کن.

In [ ]:
accuracy_from_matrix = (cm[0, 0] + cm[1, 1]) / cm.sum()
print(f"Accuracy from matrix: {accuracy_from_matrix:.1%}")

این نتیجه فقط مربوط به همین تقسیم داده است؛ تضمین عملکرد برای همهٔ مسافرها نیست. اگر مرتب براساس test مدل را تغییر بدهیم، دیگر test امتحان بی‌طرفی نیست.

## ۱۰. پیش‌بینی یک مسافر جدید

فرض کن مسافری ۲۸ ساله، زن و دارای بلیت کلاس ۲ داریم. همان Encoding قبلی را استفاده می‌کنیم: female برابر ۰. ترتیب و نام ستون‌ها باید دقیقاً مثل X_train باشد.

حتی یک مسافر هم یک جدول **2-D با شکل `(1, 3)`** می‌خواهد: یک ردیف، سه ویژگی. برای پیش‌بینی این مسافر، جواب واقعی نداریم؛ خروجی مدل یک حدس است.

In [ ]:
new_passenger = pd.DataFrame(
    [{"Pclass": 2, "Sex_encoded": 0, "Age": 28}],
    columns=feature_columns
)
# Use the training median if the new passenger's age is missing.
new_passenger["Age"] = new_passenger["Age"].fillna(age_median)
display(new_passenger)
print("Shape:", new_passenger.shape)
new_prediction = int(model.predict(new_passenger)[0])
labels = {0: "زنده نمانده", 1: "زنده مانده"}
print("Predicted label:", new_prediction)
print("پیش‌بینی مدل:", labels[new_prediction])

✏️ **تمرین آخر:** یک مسافر مرد ۴۰ ساله در کلاس ۳ بساز و پیش‌بینی کن. آیا پیش‌بینی تغییر می‌کند؟ تغییر یک ویژگی همیشه برچسب را عوض نمی‌کند. قبل از دیدن جواب، یک سلول Code اضافه کن و خودت بنویس.

In [ ]:
practice_passenger = pd.DataFrame(
    [{"Pclass": 3, "Sex_encoded": 1, "Age": 40}],
    columns=feature_columns
)
practice_passenger["Age"] = practice_passenger["Age"].fillna(age_median)
practice_prediction = int(model.predict(practice_passenger)[0])
print("Predicted label:", practice_prediction)
print("پیش‌بینی مدل:", labels[practice_prediction])

## مرور برای حمید

۱. `pd.read_csv` داده را وارد جدول کرد.
۲. `X` ویژگی‌های دوبعدی و `y` جواب‌های یک‌بعدی را نگه داشت.
۳. `train_test_split` آموزش و امتحان را جدا کرد.
۴. سن خالی را با میانهٔ train پر کردیم.
۵. `LogisticRegression` مدل را ساخت و `fit` آن را آموزش داد.
۶. `predict` برچسب را حدس زد؛ Accuracy و Confusion Matrix نتیجه را بررسی کردند.
۷. مسافر جدید را با همان ستون‌ها و همان آماده‌سازی پیش‌بینی کردیم.

**برای GitHub:** همین فایل `.ipynb` را commit کن؛ CSV برای حالت اینترنتی لازم نیست. دکمهٔ Colab به این دفترچه در `hghaderi-54/IntroAI` اشاره می‌کند. اگر فایل را جابه‌جا یا به مخزن دیگری کپی کردی، لینک دکمه را هم اصلاح کن.

منبع داده: [IntroAI — Titanic CSV](https://raw.githubusercontent.com/SharifiZarchi/IntroAI/main/Session_05/Titanic/titanic.csv). این دفترچه یک تمرین مقدماتی است و وعدهٔ رتبهٔ رقابتی در Kaggle نمی‌دهد.